## Restricted Boltzmann Machine (RBM)
This notebook provides an example of the use of Bernouilli RBM models to generate time series as proposed by [Kondratyev and Schwarz, 2020 ](https://www.risk.net/cutting-edge/7401191/the-market-generator).

### Download 4 Real Timeseries from Yahoo
Download daily historical data for Apple Inc. (AAPL) stock, the S&P 500 Index, the USD/JPY FX spot rate, and the Gold commodity spot price from 2001-01-01 to 2021-01-01

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from empirical_tests import run_tests, plot_daily_returns, acf_plots_block, qq_plot
import torch
import torch.nn as nn
import torch.nn.functional as F
from metrics import calculate_scores, real_data_loading
from timeseries import log_returns, download_data

In [ ]:
start_date = "2001-01-01"
end_date = "2021-01-01"

 # Single stock example: Apple Inc. (AAPL)
single_stock = download_data("AAPL", start_date=start_date, end_date=end_date)

# Stock index example: S&P 500 Index (^GSPC)
stock_index = download_data("^GSPC", start_date=start_date, end_date=end_date)

# FX spot example: USD/JPY (JPY=X)
fx_spot = download_data("JPY=X", start_date=start_date, end_date=end_date)

# Commodity spot example: Gold (GC=F)
commodity_spot = download_data("GC=F", start_date=start_date, end_date=end_date)

ts_lst = [single_stock['Close'].to_numpy(), stock_index['Close'].to_numpy(), fx_spot['Close'].to_numpy(), commodity_spot['Close'].to_numpy()]

single_stock_returns = log_returns(single_stock['Close'])
stock_index_returns = log_returns(stock_index['Close'])
fx_spot_returns = log_returns(fx_spot['Close'])
commodity_spot_returns = log_returns(commodity_spot['Close'])

lst_returns = [single_stock_returns, stock_index_returns, fx_spot_returns, commodity_spot_returns]
lst_labels = ['Apple', 'S&P 500', 'USDJPY', 'Gold']

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### RBM
This PyTorch class represents the RBM

In [ ]:
class BernoulliRBM(nn.Module):
    def __init__(self, visible_units, hidden_units, k=1, learning_rate=1e-3, device=device):
        super(BernoulliRBM, self).__init__()
        self.visible_units = visible_units
        self.hidden_units = hidden_units
        self.k = k
        self.learning_rate = learning_rate
        self.device = device
        
        self.W = nn.Parameter(torch.randn(visible_units, hidden_units, device=self.device) * 0.01, requires_grad=True)
        self.b_v = nn.Parameter(torch.zeros(visible_units, device=self.device), requires_grad=True)
        self.b_h = nn.Parameter(torch.zeros(hidden_units, device=self.device), requires_grad=True)

    def _sample(self, probs):
        return F.relu(torch.sign(probs - torch.rand_like(probs)))

    def contrastive_divergence(self, input_data):
        v_0 = input_data.to(self.device)
        h_0_probs = F.sigmoid(torch.matmul(v_0, self.W) + self.b_h)
        h_0 = self._sample(h_0_probs)

        for _ in range(self.k):
            v_k_probs = F.sigmoid(torch.matmul(h_0, self.W.t()) + self.b_v)
            v_k = self._sample(v_k_probs)
            h_k_probs = F.sigmoid(torch.matmul(v_k, self.W) + self.b_h)
            h_k = self._sample(h_k_probs)

        return v_0, h_0_probs, v_k, h_k_probs

    def train_step(self, input_data):
        v_0, h_0_probs, v_k, h_k_probs = self.contrastive_divergence(input_data)
        positive_grad = torch.matmul(v_0.t(), h_0_probs)
        negative_grad = torch.matmul(v_k.t(), h_k_probs)

        self.W.data += self.learning_rate * (positive_grad - negative_grad) / input_data.shape[0]
        self.b_v.data += self.learning_rate * torch.mean(v_0 - v_k, dim=0)
        self.b_h.data += self.learning_rate * torch.mean(h_0_probs - h_k_probs, dim=0)

    def train(self, train_tensor, batch_size=100, epochs=10):
        num_batches = len(train_tensor) // batch_size

        for epoch in range(epochs):
            for batch_idx in range(num_batches):
                batch = train_tensor[batch_idx * batch_size: (batch_idx + 1) * batch_size]
                self.train_step(batch)
            print(f"Epoch: {epoch + 1}/{epochs}")

    def sample(self, num_samples=1, num_gibbs_steps=1000):
        samples = torch.zeros((num_samples, self.visible_units), device=self.device)

        for i in range(num_samples):
            v_k = torch.bernoulli(torch.rand(self.visible_units)).to(self.device)
            for _ in range(num_gibbs_steps):
                h_k_probs = F.sigmoid(torch.matmul(v_k, self.W) + self.b_h)
                h_k = self._sample(h_k_probs)
                v_k_probs = F.sigmoid(torch.matmul(h_k, self.W.t()) + self.b_v)
                v_k = self._sample(v_k_probs)

            samples[i] = v_k

        return samples.detach().cpu().numpy()

The input data is a sequence of floating point returns. These need to be converted to and from 16-bit binary numbers in order to use the RBM.

In [ ]:
def quantize(arr):
    # Get the range of the input values.
    min_val = arr.min()
    max_val = arr.max()

    # Scale the input values to the range [0, 2**16 - 1].
    arr = (arr - min_val) / (max_val - min_val) * 2**16 - 32768

    # Round the scaled values to the nearest integer.
    arr = np.round(arr)

    return arr.astype(np.int16), max_val, min_val

def dequantize(arr, max_val, min_val):
    arr = arr.astype(np.float32)
    arr = (arr + 32768.0) / 65536.0 * (max_val - min_val) + min_val
    return arr

In [ ]:
quant_returns = [quantize(x) for x in lst_returns]

In [ ]:
decquant_returns = [dequantize(x[0], x[1], x[2]) for x in quant_returns]

In [ ]:
def int16_to_binary_2d(int16_array):
    binary_array = np.unpackbits(int16_array.view(np.uint8)).reshape(-1, 16)
    return binary_array

In [ ]:
def binary_2d_to_int16(binary_array):
    int16_array = np.packbits(binary_array.reshape(-1, 8)).view(np.int16)
    return int16_array

In [ ]:
bquant_returns = [int16_to_binary_2d(x[0]) for x in quant_returns]

RBM Model Parameters

In [ ]:
nv = 16
nh = 30

In [ ]:
learning_rate = 0.01
batch_size = 100
no_epochs = 10000
cd_steps = 10

Train model for Apple

In [ ]:
sim_returns = list()

In [ ]:
tbquant_returns = torch.from_numpy(bquant_returns[0]).to(device=device, dtype=torch.float32)

In [ ]:
rbm = BernoulliRBM(visible_units=nv, hidden_units=nh, k=cd_steps, learning_rate=learning_rate, device=device)
rbm.train(tbquant_returns, batch_size=batch_size, epochs=no_epochs)

In [ ]:
sample_lst = list()

for i in range(5):
    samples = rbm.sample(num_samples=1024, num_gibbs_steps=1000)
    sample_lst.append(samples)
    print('sample block {0} / 5'.format(i+1))
samples = np.concatenate(sample_lst)

In [ ]:
isamples = binary_2d_to_int16(samples[:5031].astype(np.uint8))

In [ ]:
sim_returns.append(dequantize(isamples, quant_returns[0][1], quant_returns[0][2]))

Train model for S & P 500

In [ ]:
tbquant_returns = torch.from_numpy(bquant_returns[1]).to(device=device, dtype=torch.float32)

In [ ]:
rbm = BernoulliRBM(visible_units=nv, hidden_units=nh, k=cd_steps, learning_rate=learning_rate, device=device)
rbm.train(tbquant_returns, batch_size=batch_size, epochs=no_epochs)

In [ ]:
sample_lst = list()

for i in range(5):
    samples = rbm.sample(num_samples=1024, num_gibbs_steps=1000)
    sample_lst.append(samples)
    print('sample block {0} / 5'.format(i+1))
samples = np.concatenate(sample_lst)
isamples = binary_2d_to_int16(samples[:5031].astype(np.uint8))
sim_returns.append(dequantize(isamples, quant_returns[1][1], quant_returns[1][2]))

Train model for USDJPY

In [ ]:
tbquant_returns = torch.from_numpy(bquant_returns[2]).to(device=device, dtype=torch.float32)

In [ ]:
rbm = BernoulliRBM(visible_units=nv, hidden_units=nh, k=cd_steps, learning_rate=learning_rate, device=device)
rbm.train(tbquant_returns, batch_size=batch_size, epochs=no_epochs)

In [ ]:
sample_lst = list()

for i in range(5):
    samples = rbm.sample(num_samples=1024, num_gibbs_steps=1000)
    sample_lst.append(samples)
    print('sample block {0} / 5'.format(i+1))
samples = np.concatenate(sample_lst)
isamples = binary_2d_to_int16(samples[:5031].astype(np.uint8))
sim_returns.append(dequantize(isamples, quant_returns[2][1], quant_returns[2][2]))

Train model for Gold

In [ ]:
tbquant_returns = torch.from_numpy(bquant_returns[3]).to(device=device, dtype=torch.float32)

In [ ]:
rbm = BernoulliRBM(visible_units=nv, hidden_units=nh, k=cd_steps, learning_rate=learning_rate, device=device)
rbm.train(tbquant_returns, batch_size=batch_size, epochs=no_epochs)

In [ ]:
sample_lst = list()

for i in range(5):
    samples = rbm.sample(num_samples=1024, num_gibbs_steps=1000)
    sample_lst.append(samples)
    print('sample block {0} / 5'.format(i+1))
samples = np.concatenate(sample_lst)
isamples = binary_2d_to_int16(samples[:5031].astype(np.uint8))
sim_returns.append(dequantize(isamples, quant_returns[3][1], quant_returns[3][2]))

#### Daily Returns

In [ ]:
rbm_filename = 'RBMReturns.png'
plot_daily_returns(sim_returns, stock_index.index[:-1], rbm_filename, lst_labels)

#### Run Empirical Tests

In [ ]:
df = run_tests(sim_returns, lst_labels)
df = df.set_index('Label')
pd.set_option('display.precision', 2)
df

In [ ]:
df.to_latex()

#### Plot ACF for Returns and Squared Returns

In [ ]:
rbm_filename = 'RBMACFs.png'
acf_plots_block(sim_returns, lst_labels, rbm_filename)

#### QQ Plot

In [ ]:
rbm_qq_filename = 'RBMQQ.png'
qq_plot(lst_returns, sim_returns, rbm_qq_filename, num_cols=2, labels=lst_labels)

#### Predictive and Discriminative Scores

In [ ]:
seq_len = 1

In [ ]:
ssr2 = real_data_loading(single_stock_returns, 32)
sir2 = real_data_loading(stock_index_returns, 32)
fxr2 = real_data_loading(fx_spot_returns, 32)
csr2 = real_data_loading(commodity_spot_returns, 32)

In [ ]:
sim_ssr2 = real_data_loading(sim_returns[0], 32)
sim_sir2 = real_data_loading(sim_returns[1], 32)
sim_fxr2 = real_data_loading(sim_returns[2], 32)
sim_csr2 = real_data_loading(sim_returns[3], 32)

In [ ]:
real_lst = [ssr2, sir2, fxr2, csr2]
sim_lst = [sim_ssr2, sim_sir2, sim_fxr2, sim_csr2]

In [ ]:
sim_returns

In [ ]:
import numpy as np
import csv

# Loop through the arrays and names, and write each array to a CSV file
for i, arr in enumerate(sim_returns):
    filename = lst_labels[i] + '.csv'
    with open(filename, 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Index', 'Value'])
        for j, val in enumerate(arr):
            writer.writerow([j, val])


In [ ]:
import csv

def read_csv_files(names):
    arrays = {}
    for name in names:
        filename = name + '.csv'
        with open(filename, 'r', newline='') as file:
            reader = csv.reader(file)
            header = next(reader)  # Skip the header row
            data = []
            for row in reader:
                data.append(float(row[1]))
            arrays[name] = np.array(data)
    return arrays

In [ ]:
arr_dict = read_csv_files(lst_labels)

In [ ]:
arr_dict

In [ ]:
ssr2 = real_data_loading(single_stock_returns, 32)
sir2 = real_data_loading(stock_index_returns, 32)
fxr2 = real_data_loading(fx_spot_returns, 32)
csr2 = real_data_loading(commodity_spot_returns, 32)

In [ ]:
ssrsyn = real_data_loading(arr_dict['Apple'], 32)
sirsyn = real_data_loading(arr_dict['S&P 500'], 32)
fxrsyn = real_data_loading(arr_dict['USDJPY'], 32)
csrsyn = real_data_loading(arr_dict['Gold'], 32)

In [ ]:
real_lst = [ssr2, sir2, fxr2, csr2]
sim_lst = [ssrsyn, sirsyn, fxrsyn, csrsyn]

In [ ]:
score_df = calculate_scores(real_lst, sim_lst, lst_labels, device)

In [ ]:
score_df

In [ ]:
score_df.to_latex()